In [1]:
from util import import_ragas_custom, load_env_variables_from_all_env_files

import_ragas_custom('ragas_custom_2')
load_env_variables_from_all_env_files()

Arquivos copiados com sucesso!


In [2]:
import os
import asyncio
import nest_asyncio

from ragas.integrations.llama_index import evaluate
from ragas.run_config import RunConfig
from ragas.testset.synthesizers.testset_schema import Testset
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding, OpenAIEmbeddingModelType
from llama_index.llms.llama_api import LlamaAPI

from ragas.prompt.mixin import PromptMixin
from ragas.run_config import RunConfig

from llama_index.core import (
    VectorStoreIndex,
    SimpleDirectoryReader,
    Settings,
    StorageContext,
    load_index_from_storage,
)

from ragas.metrics import (
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity,
    ResponseRelevancy,
    answer_relevancy,
    faithfulness,
    FactualCorrectness,
    SemanticSimilarity,
    NonLLMStringSimilarity,
    RougeScore,
    StringPresence,
    ExactMatch
)

from ragas.metrics._aspect_critic import SUPPORTED_ASPECTS

[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\jmess\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [3]:
nest_asyncio.apply()

In [4]:
DATA_PATH = 'data'
TESTSET = 'testset_openai_4omini.jsonl'
LANGUAGE = 'portuguese'
TIMEOUT = 300
RESULT_CSV = 'result_gpt4omini_llama3.2_1b'
MODEL_OLLAMA = 'llama3.2:3b'
MODEL_GPT = 'gpt-4o-mini-2024-07-18'
MODEL_GPT_EMBEDING = OpenAIEmbeddingModelType.TEXT_EMBED_3_SMALL
CACHE_DIR = 'cache'
MODEL_LLAMA_API = 'llama3.2-1b'
PERSIST_DIR = "./cache_gpt_emb_small"

In [5]:
run_config = RunConfig(timeout=TIMEOUT, max_workers=1)

In [6]:
testset = Testset.from_jsonl(TESTSET).to_pandas()

print("Tamanho do dataset: ", len(testset))
print(testset.head())

Tamanho do dataset:  132
                                          user_input  \
0  Quais são as aplicações e benefícios dos No-Br...   
1  Quais são as aplicações e benefícios dos No-Br...   
2  impacto da proteção contra distúrbios de energ...   
3  De que forma a protecão contra distúrbios de e...   
4  implicações suporte técnico eficaz melhoria ex...   

                                  reference_contexts  \
0  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1  [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2  [plicações de  \nmissão crítica, nas mais vari...   
3  [plicações de  \nmissão crítica, nas mais vari...   
4  [supor te técnico pode oferecer ., supor te té...   

                                           reference          synthesizer_name  
0  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
1  Os No-Breaks da CM Comandos são indicados para...  AbstractQuerySynthesizer  
2  A proteção contra distúrbios de energia elétri...  Abst

In [7]:
nan_rows = testset[testset.isna().any(axis=1)]

print("Quantidade de nulos: ", len(nan_rows))
print(nan_rows)

del nan_rows

Quantidade de nulos:  0
Empty DataFrame
Columns: [user_input, reference_contexts, reference, synthesizer_name]
Index: []


In [8]:
testset = testset.dropna()

In [9]:
print("Quantidade de linhas após remoção de nulos: ", len(testset))
print(testset)

Quantidade de linhas após remoção de nulos:  132
                                            user_input  \
0    Quais são as aplicações e benefícios dos No-Br...   
1    Quais são as aplicações e benefícios dos No-Br...   
2    impacto da proteção contra distúrbios de energ...   
3    De que forma a protecão contra distúrbios de e...   
4    implicações suporte técnico eficaz melhoria ex...   
..                                                 ...   
127  Qual é a importância do termo 'BEN' no context...   
128  Qual é a função do disjuntor no processo de ma...   
129  Quais são as aplicações mais modernas do DSP n...   
130  Quais são as vantagens do modelo cliente-serve...   
131                        significado de 'Po' lti-S.O   

                                    reference_contexts  \
0    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
1    [A EMPRESA\nOs    No-Breaks   da   CM     Coma...   
2    [plicações de  \nmissão crítica, nas mais vari...   
3    [plicações de  \n

In [10]:
base = (len(testset) / 5)
index_csv= 4
min = int((index_csv - 1) * base )
max = int(index_csv * base)
testset = Testset.from_pandas(testset[104:105])
print(f'min: {min} | max: {max-1}')

min: 79 | max: 104


In [11]:
# embeding = OpenAIEmbedding(model=MODEL_GPT_EMBEDING)
# model = OpenAI(model=MODEL_GPT)

model = LlamaAPI(model=MODEL_LLAMA_API, api_key=os.getenv('LLAMA_API_KEY'))
embeding = OpenAIEmbedding(model=MODEL_GPT_EMBEDING)

avaliator_llm = OpenAI(model=MODEL_GPT)
# avaliator_llm = model

Settings.embed_model = embeding
Settings.llm = model

In [12]:
metrics = [
    context_precision,
    context_recall,
    context_entity_recall,
    NoiseSensitivity(),
    ResponseRelevancy(),
    answer_relevancy,
    faithfulness,
    FactualCorrectness(),
    SemanticSimilarity(),
    NonLLMStringSimilarity(),
    RougeScore(),
    StringPresence(),
    ExactMatch()
]

metrics.extend(SUPPORTED_ASPECTS)

for query in metrics:
    if isinstance(query, PromptMixin):
        path = os.path.join(CACHE_DIR, query.__class__.__name__)
        if not os.path.exists(path):
            os.makedirs(path)

        try:
            prompts = query.load_prompts(path, LANGUAGE,)
            query.set_prompts(**prompts)
        except Exception:
            prompts = asyncio.run(query.adapt_prompts(LANGUAGE, None, True, True))
            query.set_prompts(**prompts)
            query.save_prompts(path)
            prompts = query.load_prompts(path, LANGUAGE)
            query.set_prompts(**prompts)


In [13]:
if not os.path.exists(PERSIST_DIR):
    documents = SimpleDirectoryReader(DATA_PATH).load_data()
    index = VectorStoreIndex.from_documents(documents)
    index.storage_context.persist(persist_dir=PERSIST_DIR)
else:
    storage_context = StorageContext.from_defaults(persist_dir=PERSIST_DIR)
    index = load_index_from_storage(storage_context)
    
query_engine = index.as_query_engine(request_timeout=TIMEOUT)

In [14]:
model.complete("test")

CompletionResponse(text="It seems like you're testing or trying out our conversation. That's perfectly fine. If you have any questions or need assistance with something, feel free to ask.", additional_kwargs={'refusal': None, 'audio': None, 'function_call': None, 'tool_calls': None}, raw={'created': 1737645587, 'model': 'llama3.2-1b', 'usage': {'prompt_tokens': 63, 'completion_tokens': 56, 'total_tokens': 119}, 'choices': [{'finish_reason': 'stop', 'index': 0, 'logprobs': None, 'message': {'content': "It seems like you're testing or trying out our conversation. That's perfectly fine. If you have any questions or need assistance with something, feel free to ask.", 'refusal': None, 'role': 'assistant', 'audio': None, 'function_call': None, 'tool_calls': None}}]}, logprobs=None, delta=None)

In [15]:
query_engine.query("teste")

Response(response='A tecnologia utilizada para realizar medições em True RMS em frequências rápidas é baseada em amostragem digital.', source_nodes=[NodeWithScore(node=TextNode(id_='7c723bb1-cbd8-41db-8a15-e36354fdb268', embedding=None, metadata={'page_label': '1', 'file_name': 'catalogo-prevention-site.pdf', 'file_path': 'd:\\GitHub\\rag_eval\\data\\catalogo-prevention-site.pdf', 'file_type': 'application/pdf', 'file_size': 3542656, 'creation_date': '2024-10-09', 'last_modified_date': '2024-10-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='31e31c15-7669-4517-8678-f51553e964cb', node_type='4', metadata={'page_label': '1', 'file_name': 'catalogo-prevention-site.pdf', 'file_path': 'd:\\GitHub\\rag_eval

In [16]:
asyncio.run(query_engine.aquery("teste"))

Response(response='A tecnologia utilizada para realizar medições em True RMS em frequências rápidas é baseada em amostragem digital.', source_nodes=[NodeWithScore(node=TextNode(id_='7c723bb1-cbd8-41db-8a15-e36354fdb268', embedding=None, metadata={'page_label': '1', 'file_name': 'catalogo-prevention-site.pdf', 'file_path': 'd:\\GitHub\\rag_eval\\data\\catalogo-prevention-site.pdf', 'file_type': 'application/pdf', 'file_size': 3542656, 'creation_date': '2024-10-09', 'last_modified_date': '2024-10-09'}, excluded_embed_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], excluded_llm_metadata_keys=['file_name', 'file_type', 'file_size', 'creation_date', 'last_modified_date', 'last_accessed_date'], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='31e31c15-7669-4517-8678-f51553e964cb', node_type='4', metadata={'page_label': '1', 'file_name': 'catalogo-prevention-site.pdf', 'file_path': 'd:\\GitHub\\rag_eval

In [17]:
result = evaluate(
    query_engine=query_engine,
    metrics=metrics,
    dataset=testset,
    llm=avaliator_llm,
    embeddings=embeding,
    run_config=run_config
)

Running Query Engine:   0%|          | 0/25 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/450 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[150]: AttributeError('StringIO' object has no attribute 'statements')
ERROR:ragas.executor:Exception raised in Job[183]: AttributeError('StringIO' object has no attribute 'statements')


In [18]:
result_dataframe = result.to_pandas()
result_dataframe.to_csv(f'{RESULT_CSV}_{6}.csv')

In [19]:
print(result)

{'context_precision': 0.5000, 'context_recall': 0.3200, 'context_entity_recall': 0.0866, 'noise_sensitivity_relevant': 0.1016, 'answer_relevancy': 0.8061, 'faithfulness': 0.2619, 'factual_correctness': 0.2172, 'semantic_similarity': 0.7380, 'non_llm_string_similarity': 0.2296, 'rouge_score': 0.2472, 'string_present': 0.0000, 'exact_match': 0.0000, 'harmfulness': 0.3600, 'maliciousness': 0.4000, 'coherence': 0.8800, 'correctness': 0.6000, 'conciseness': 0.4800}
